## 1.DummyClassifier

In [1]:
from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
)

warnings.filterwarnings("ignore")

In [2]:
# 定位项目路径并创建输出目录
CURRENT_DIR = Path.cwd()

if (CURRENT_DIR / "data").exists():
    PROJECT_DIR = CURRENT_DIR
elif (CURRENT_DIR.parent / "data").exists():
    PROJECT_DIR = CURRENT_DIR.parent
else:
    raise FileNotFoundError(
        "Cannot locate project root. Please run this notebook inside the project folder or notebook folder."
    )

DATA_DIR = PROJECT_DIR / "data" / "raw"

OUTPUT_DIR = PROJECT_DIR / "reports" / "model" / "baseline" / "dummy"
TABLE_DIR = OUTPUT_DIR / "tables"
FIGURE_DIR = OUTPUT_DIR / "figures"

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("PROJECT_DIR:", PROJECT_DIR)
print("DATA_DIR:", DATA_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)

PROJECT_DIR: d:\A_projects\kaggle-customer-churn-prediction
DATA_DIR: d:\A_projects\kaggle-customer-churn-prediction\data\raw
OUTPUT_DIR: d:\A_projects\kaggle-customer-churn-prediction\reports\model\baseline\dummy


In [3]:
train_path = DATA_DIR / "train.csv"
test_path = DATA_DIR / "test.csv"
sample_submission_path = DATA_DIR / "sample_submission.csv"

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_submission = pd.read_csv(sample_submission_path)

print("train shape:", train.shape)
print("test shape:", test.shape)
print("sample submission shape:", sample_submission.shape)

display(train.head())

train shape: (594194, 21)
test shape: (254655, 20)
sample submission shape: (254655, 2)


,id,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,...,Yes,Yes,No,No,One year,Yes,Mailed check,60.10,1653.85,No
1,1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,...,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.50,3778.20,No
2,2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,...,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.40,5841.35,No
3,3,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,69.70,70.70,Yes
4,4,Female,0,No,No,1,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.45,70.45,Yes


In [4]:
#划分特征与目标变量
TARGET_COL = "Churn"

if TARGET_COL not in train.columns:
    raise ValueError(f"Target column '{TARGET_COL}' not found in train.csv")

id_candidates = {
    "id",
    "customerid",
    "customer_id",
    "clientid",
    "client_id",
    "userid",
    "user_id",
}

id_cols = [col for col in train.columns if col.lower() in id_candidates]

feature_cols = [col for col in train.columns if col not in id_cols + [TARGET_COL]]

X = train[feature_cols]
y = train[TARGET_COL]

print("ID columns:", id_cols)
print("Target column:", TARGET_COL)
print("Number of features:", len(feature_cols))
print("Target distribution:")
display(y.value_counts(normalize=True).rename("proportion").to_frame())

ID columns: ['id']
Target column: Churn
Number of features: 19
Target distribution:


,proportion
Churn,
No,0.774792
Yes,0.225208


In [5]:
#划分训练集与验证集
RANDOM_STATE = 42
TEST_SIZE = 0.2

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("X_train shape:", X_train.shape)
print("X_valid shape:", X_valid.shape)

print("Train target distribution:")
display(y_train.value_counts(normalize=True).rename("proportion").to_frame())

print("Valid target distribution:")
display(y_valid.value_counts(normalize=True).rename("proportion").to_frame())

X_train shape: (475355, 19)
X_valid shape: (118839, 19)
Train target distribution:


,proportion
Churn,
No,0.774791
Yes,0.225209


Valid target distribution:


,proportion
Churn,
No,0.774796
Yes,0.225204


In [6]:
#训练基线模型并评估性能
dummy_model = DummyClassifier(
    strategy="most_frequent",
    random_state=RANDOM_STATE,
)

dummy_model.fit(X_train, y_train)

y_valid_pred = dummy_model.predict(X_valid)

print("Dummy model classes:", dummy_model.classes_)
print("Predicted class distribution:")
display(pd.Series(y_valid_pred).value_counts(normalize=True).rename("proportion").to_frame())

Dummy model classes: ['No' 'Yes']
Predicted class distribution:


,proportion
No,1.0


In [7]:
#计算验证集指标
classes = dummy_model.classes_

if len(classes) != 2:
    raise ValueError("This notebook assumes a binary classification task.")

# 自动识别正类
if 1 in classes:
    positive_class = 1
elif "Yes" in classes:
    positive_class = "Yes"
elif "yes" in classes:
    positive_class = "yes"
elif "True" in classes:
    positive_class = "True"
else:
    positive_class = classes[1]

positive_index = np.where(classes == positive_class)[0][0]

y_valid_binary = (y_valid == positive_class).astype(int)

if hasattr(dummy_model, "predict_proba"):
    y_valid_proba = dummy_model.predict_proba(X_valid)[:, positive_index]
else:
    y_valid_proba = None

metrics = {
    "model": "DummyClassifier",
    "strategy": "most_frequent",
    "positive_class": positive_class,
    "train_size": len(X_train),
    "valid_size": len(X_valid),
    "accuracy": accuracy_score(y_valid, y_valid_pred),
    "balanced_accuracy": balanced_accuracy_score(y_valid, y_valid_pred),
    "precision": precision_score(y_valid, y_valid_pred, pos_label=positive_class, zero_division=0),
    "recall": recall_score(y_valid, y_valid_pred, pos_label=positive_class, zero_division=0),
    "f1": f1_score(y_valid, y_valid_pred, pos_label=positive_class, zero_division=0),
}

if y_valid_proba is not None:
    metrics["roc_auc"] = roc_auc_score(y_valid_binary, y_valid_proba)
    metrics["average_precision"] = average_precision_score(y_valid_binary, y_valid_proba)

metrics_df = pd.DataFrame([metrics])

display(metrics_df)

,model,strategy,positive_class,train_size,valid_size,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,average_precision
0,DummyClassifier,most_frequent,Yes,475355,118839,0.774796,0.5,0.0,0.0,0.0,0.5,0.225204


In [8]:
#保存分类报告和混淆矩阵
report_dict = classification_report(
    y_valid,
    y_valid_pred,
    output_dict=True,
    zero_division=0,
)

classification_report_df = pd.DataFrame(report_dict).T

cm = confusion_matrix(y_valid, y_valid_pred, labels=classes)
confusion_matrix_df = pd.DataFrame(
    cm,
    index=[f"actual_{cls}" for cls in classes],
    columns=[f"pred_{cls}" for cls in classes],
)

metrics_df.to_csv(TABLE_DIR / "dummy_metrics.csv", index=False)
classification_report_df.to_csv(TABLE_DIR / "dummy_classification_report.csv")
confusion_matrix_df.to_csv(TABLE_DIR / "dummy_confusion_matrix.csv")

display(classification_report_df)
display(confusion_matrix_df)

print("Saved tables to:", TABLE_DIR)

,precision,recall,f1-score,support
No,0.774796,1.000000,0.873110,92076.000000
Yes,0.000000,0.000000,0.000000,26763.000000
accuracy,0.774796,0.774796,0.774796,0.774796
macro avg,0.387398,0.500000,0.436555,118839.000000
weighted avg,0.600309,0.774796,0.676482,118839.000000


,pred_No,pred_Yes
actual_No,92076,0
actual_Yes,26763,0


Saved tables to: d:\A_projects\kaggle-customer-churn-prediction\reports\model\baseline\dummy\tables


## 2. LogisticRegression Baseline

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

In [10]:
LOGISTIC_OUTPUT_DIR = PROJECT_DIR / "reports" / "model" / "baseline" / "logistic_regression"
LOGISTIC_TABLE_DIR = LOGISTIC_OUTPUT_DIR / "tables"
LOGISTIC_FIGURE_DIR = LOGISTIC_OUTPUT_DIR / "figures"

LOGISTIC_TABLE_DIR.mkdir(parents=True, exist_ok=True)
LOGISTIC_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("LOGISTIC_OUTPUT_DIR:", LOGISTIC_OUTPUT_DIR)

LOGISTIC_OUTPUT_DIR: d:\A_projects\kaggle-customer-churn-prediction\reports\model\baseline\logistic_regression


In [11]:
#识别数值型和类别型特征
numeric_cols = X_train.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
categorical_cols = X_train.select_dtypes(include=["object", "category", "string", "bool"]).columns.tolist()

print("Number of numeric columns:", len(numeric_cols))
print("Numeric columns:", numeric_cols)

print("Number of categorical columns:", len(categorical_cols))
print("Categorical columns:", categorical_cols)

Number of numeric columns: 4
Numeric columns: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
Number of categorical columns: 15
Categorical columns: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


In [12]:
#构建预处理器和LogisticRegression模型
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols),
    ],
    remainder="drop",
)

logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                solver="lbfgs",
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

logistic_model

UnicodeDecodeError: 'gbk' codec can't decode byte 0x94 in position 2952: illegal multibyte sequence

UnicodeDecodeError: 'gbk' codec can't decode byte 0x94 in position 2952: illegal multibyte sequence

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['SeniorCitizen', 'tenure',
                                                   'MonthlyCharges',
                                                   'TotalCharges']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                            

In [13]:
#训练LogisticRegression模型
logistic_model.fit(X_train, y_train)

print("LogisticRegression training completed.")

LogisticRegression training completed.


In [14]:
#验证集预测
y_valid_pred_logistic = logistic_model.predict(X_valid)

classes = logistic_model.named_steps["model"].classes_

if 1 in classes:
    positive_class = 1
elif "Yes" in classes:
    positive_class = "Yes"
elif "yes" in classes:
    positive_class = "yes"
elif "True" in classes:
    positive_class = "True"
else:
    positive_class = classes[1]

positive_index = np.where(classes == positive_class)[0][0]

y_valid_proba_logistic = logistic_model.predict_proba(X_valid)[:, positive_index]
y_valid_binary = (y_valid == positive_class).astype(int)

print("Model classes:", classes)
print("Positive class:", positive_class)

print("Predicted class distribution:")
display(
    pd.Series(y_valid_pred_logistic)
    .value_counts(normalize=True)
    .rename("proportion")
    .to_frame()
)

Model classes: ['No' 'Yes']
Positive class: Yes
Predicted class distribution:


,proportion
No,0.783253
Yes,0.216747


In [15]:
# 计算逻辑回归模型的评估指标
logistic_metrics = {
    "model": "LogisticRegression",
    "positive_class": positive_class,
    "train_size": len(X_train),
    "valid_size": len(X_valid),
    "accuracy": accuracy_score(y_valid, y_valid_pred_logistic),
    "balanced_accuracy": balanced_accuracy_score(y_valid, y_valid_pred_logistic),
    "precision": precision_score(
        y_valid,
        y_valid_pred_logistic,
        pos_label=positive_class,
        zero_division=0,
    ),
    "recall": recall_score(
        y_valid,
        y_valid_pred_logistic,
        pos_label=positive_class,
        zero_division=0,
    ),
    "f1": f1_score(
        y_valid,
        y_valid_pred_logistic,
        pos_label=positive_class,
        zero_division=0,
    ),
    "roc_auc": roc_auc_score(y_valid_binary, y_valid_proba_logistic),
    "average_precision": average_precision_score(y_valid_binary, y_valid_proba_logistic),
}

logistic_metrics_df = pd.DataFrame([logistic_metrics])

display(logistic_metrics_df)

,model,positive_class,train_size,valid_size,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,average_precision
0,LogisticRegression,Yes,475355,118839,0.854416,0.784753,0.683671,0.657998,0.670589,0.908428,0.727093


In [16]:
#保存LogisticRegression分类报告和混淆矩阵
logistic_report_dict = classification_report(
    y_valid,
    y_valid_pred_logistic,
    output_dict=True,
    zero_division=0,
)

logistic_classification_report_df = pd.DataFrame(logistic_report_dict).T

logistic_cm = confusion_matrix(
    y_valid,
    y_valid_pred_logistic,
    labels=classes,
)

logistic_confusion_matrix_df = pd.DataFrame(
    logistic_cm,
    index=[f"actual_{cls}" for cls in classes],
    columns=[f"pred_{cls}" for cls in classes],
)

logistic_metrics_df.to_csv(
    LOGISTIC_TABLE_DIR / "logistic_metrics.csv",
    index=False,
)

logistic_classification_report_df.to_csv(
    LOGISTIC_TABLE_DIR / "logistic_classification_report.csv",
    index_label="class",
)

logistic_confusion_matrix_df.to_csv(
    LOGISTIC_TABLE_DIR / "logistic_confusion_matrix.csv",
)

display(logistic_classification_report_df)
display(logistic_confusion_matrix_df)

print("Saved LogisticRegression tables to:", LOGISTIC_TABLE_DIR)

,precision,recall,f1-score,support
No,0.901666,0.911508,0.906560,92076.000000
Yes,0.683671,0.657998,0.670589,26763.000000
accuracy,0.854416,0.854416,0.854416,0.854416
macro avg,0.792669,0.784753,0.788575,118839.000000
weighted avg,0.852573,0.854416,0.853419,118839.000000


,pred_No,pred_Yes
actual_No,83928,8148
actual_Yes,9153,17610


Saved LogisticRegression tables to: d:\A_projects\kaggle-customer-churn-prediction\reports\model\baseline\logistic_regression\tables


In [17]:
#提取LogisticRegression模型的特征重要性（系数）并保存
fitted_preprocessor = logistic_model.named_steps["preprocessor"]
fitted_logistic = logistic_model.named_steps["model"]

feature_names = fitted_preprocessor.get_feature_names_out()
coefficients = fitted_logistic.coef_[0]

coef_df = pd.DataFrame(
    {
        "feature": feature_names,
        "coefficient": coefficients,
        "abs_coefficient": np.abs(coefficients),
    }
)

coef_df = coef_df.sort_values("abs_coefficient", ascending=False)

coef_df.to_csv(
    LOGISTIC_TABLE_DIR / "logistic_feature_coefficients.csv",
    index=False,
)

display(coef_df.head(30))

,feature,coefficient,abs_coefficient
1,num__tenure,-1.827211,1.827211
38,cat__Contract_Two year,-0.831312,0.831312
3,num__TotalCharges,0.817186,0.817186
36,cat__Contract_Month-to-month,0.565602,0.565602
15,cat__InternetService_DSL,-0.499677,0.499677
39,cat__PaperlessBilling_No,-0.443150,0.443150
11,cat__PhoneService_Yes,-0.432928,0.432928
43,cat__PaymentMethod_Electronic check,0.412822,0.412822
9,cat__Dependents_Yes,-0.393405,0.393405
12,cat__MultipleLines_No,-0.364732,0.364732


In [18]:
#找到最重要的正向和负向特征
top_positive_coef = coef_df.sort_values("coefficient", ascending=False).head(20)
top_negative_coef = coef_df.sort_values("coefficient", ascending=True).head(20)

top_positive_coef.to_csv(
    LOGISTIC_TABLE_DIR / "logistic_top_positive_coefficients.csv",
    index=False,
)

top_negative_coef.to_csv(
    LOGISTIC_TABLE_DIR / "logistic_top_negative_coefficients.csv",
    index=False,
)

print("Top positive coefficients:")
display(top_positive_coef)

print("Top negative coefficients:")
display(top_negative_coef)

Top positive coefficients:


,feature,coefficient,abs_coefficient
3,num__TotalCharges,0.817186,0.817186
36,cat__Contract_Month-to-month,0.565602,0.565602
43,cat__PaymentMethod_Electronic check,0.412822,0.412822
16,cat__InternetService_Fiber optic,0.288246,0.288246
2,num__MonthlyCharges,0.223771,0.223771
0,num__SeniorCitizen,0.138463,0.138463
18,cat__OnlineSecurity_No,0.099625,0.099625
32,cat__StreamingTV_Yes,0.031213,0.031213
27,cat__TechSupport_No,0.023540,0.023540
35,cat__StreamingMovies_Yes,0.018451,0.018451


Top negative coefficients:


,feature,coefficient,abs_coefficient
1,num__tenure,-1.827211,1.827211
38,cat__Contract_Two year,-0.831312,0.831312
15,cat__InternetService_DSL,-0.499677,0.499677
39,cat__PaperlessBilling_No,-0.443150,0.443150
11,cat__PhoneService_Yes,-0.432928,0.432928
9,cat__Dependents_Yes,-0.393405,0.393405
12,cat__MultipleLines_No,-0.364732,0.364732
44,cat__PaymentMethod_Mailed check,-0.316218,0.316218
20,cat__OnlineSecurity_Yes,-0.311056,0.311056
42,cat__PaymentMethod_Credit card (automatic),-0.284133,0.284133


## 3. RandomForest Baseline

In [19]:
from sklearn.ensemble import RandomForestClassifier

In [20]:
RF_OUTPUT_DIR = PROJECT_DIR / "reports" / "model" / "baseline" / "random_forest"
RF_TABLE_DIR = RF_OUTPUT_DIR / "tables"
RF_FIGURE_DIR = RF_OUTPUT_DIR / "figures"

RF_TABLE_DIR.mkdir(parents=True, exist_ok=True)
RF_FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("RF_OUTPUT_DIR:", RF_OUTPUT_DIR)

RF_OUTPUT_DIR: d:\A_projects\kaggle-customer-churn-prediction\reports\model\baseline\random_forest


In [21]:
#构建RandomForest预处理器
rf_numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
    ]
)

rf_categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

rf_preprocessor = ColumnTransformer(
    transformers=[
        ("num", rf_numeric_transformer, numeric_cols),
        ("cat", rf_categorical_transformer, categorical_cols),
    ],
    remainder="drop",
)

In [22]:
#构建RandomForest模型
rf_model = Pipeline(
    steps=[
        ("preprocessor", rf_preprocessor),
        (
            "model",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=None,
                min_samples_split=2,
                min_samples_leaf=1,
                max_features="sqrt",
                random_state=RANDOM_STATE,
                n_jobs=-1,
                class_weight=None,
            ),
        ),
    ]
)

rf_model

UnicodeDecodeError: 'gbk' codec can't decode byte 0x94 in position 2952: illegal multibyte sequence

UnicodeDecodeError: 'gbk' codec can't decode byte 0x94 in position 2952: illegal multibyte sequence

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  ['SeniorCitizen', 'tenure',
                                                   'MonthlyCharges',
                                                   'TotalCharges']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  ['gender', 'Partner',
         

In [23]:
#训练RandomForest模型
rf_model.fit(X_train, y_train)

print("RandomForest training completed.")

RandomForest training completed.


In [24]:
#预测验证集上的结果
y_valid_pred_rf = rf_model.predict(X_valid)

rf_classes = rf_model.named_steps["model"].classes_

if 1 in rf_classes:
    rf_positive_class = 1
elif "Yes" in rf_classes:
    rf_positive_class = "Yes"
elif "yes" in rf_classes:
    rf_positive_class = "yes"
elif "True" in rf_classes:
    rf_positive_class = "True"
else:
    rf_positive_class = rf_classes[1]

rf_positive_index = np.where(rf_classes == rf_positive_class)[0][0]

y_valid_proba_rf = rf_model.predict_proba(X_valid)[:, rf_positive_index]
y_valid_binary_rf = (y_valid == rf_positive_class).astype(int)

print("Model classes:", rf_classes)
print("Positive class:", rf_positive_class)

print("Predicted class distribution:")
display(
    pd.Series(y_valid_pred_rf)
    .value_counts(normalize=True)
    .rename("proportion")
    .to_frame()
)

Model classes: ['No' 'Yes']
Positive class: Yes
Predicted class distribution:


,proportion
No,0.79829
Yes,0.20171


In [25]:
#计算RandomForest模型的评估指标
rf_metrics = {
    "model": "RandomForest",
    "positive_class": rf_positive_class,
    "train_size": len(X_train),
    "valid_size": len(X_valid),
    "accuracy": accuracy_score(y_valid, y_valid_pred_rf),
    "balanced_accuracy": balanced_accuracy_score(y_valid, y_valid_pred_rf),
    "precision": precision_score(
        y_valid,
        y_valid_pred_rf,
        pos_label=rf_positive_class,
        zero_division=0,
    ),
    "recall": recall_score(
        y_valid,
        y_valid_pred_rf,
        pos_label=rf_positive_class,
        zero_division=0,
    ),
    "f1": f1_score(
        y_valid,
        y_valid_pred_rf,
        pos_label=rf_positive_class,
        zero_division=0,
    ),
    "roc_auc": roc_auc_score(y_valid_binary_rf, y_valid_proba_rf),
    "average_precision": average_precision_score(y_valid_binary_rf, y_valid_proba_rf),
}

rf_metrics_df = pd.DataFrame([rf_metrics])

display(rf_metrics_df)

,model,positive_class,train_size,valid_size,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,average_precision
0,RandomForest,Yes,475355,118839,0.840423,0.752862,0.662676,0.593543,0.626207,0.892798,0.69061


In [26]:
#保存RandomForest分类报告和混淆矩阵
rf_report_dict = classification_report(
    y_valid,
    y_valid_pred_rf,
    output_dict=True,
    zero_division=0,
)

rf_classification_report_df = pd.DataFrame(rf_report_dict).T

rf_cm = confusion_matrix(
    y_valid,
    y_valid_pred_rf,
    labels=rf_classes,
)

rf_confusion_matrix_df = pd.DataFrame(
    rf_cm,
    index=[f"actual_{cls}" for cls in rf_classes],
    columns=[f"pred_{cls}" for cls in rf_classes],
)

rf_metrics_df.to_csv(
    RF_TABLE_DIR / "random_forest_metrics.csv",
    index=False,
)

rf_classification_report_df.to_csv(
    RF_TABLE_DIR / "random_forest_classification_report.csv",
    index_label="class",
)

rf_confusion_matrix_df.to_csv(
    RF_TABLE_DIR / "random_forest_confusion_matrix.csv",
)

display(rf_classification_report_df)
display(rf_confusion_matrix_df)

print("Saved RandomForest tables to:", RF_TABLE_DIR)

,precision,recall,f1-score,support
No,0.885335,0.912181,0.898558,92076.000000
Yes,0.662676,0.593543,0.626207,26763.000000
accuracy,0.840423,0.840423,0.840423,0.840423
macro avg,0.774006,0.752862,0.762383,118839.000000
weighted avg,0.835192,0.840423,0.837223,118839.000000


,pred_No,pred_Yes
actual_No,83990,8086
actual_Yes,10878,15885


Saved RandomForest tables to: d:\A_projects\kaggle-customer-churn-prediction\reports\model\baseline\random_forest\tables


In [27]:
#保存RandomForest特征重要性
rf_fitted_preprocessor = rf_model.named_steps["preprocessor"]
rf_fitted_model = rf_model.named_steps["model"]

rf_feature_names = rf_fitted_preprocessor.get_feature_names_out()
rf_feature_importances = rf_fitted_model.feature_importances_

rf_feature_importance_df = pd.DataFrame(
    {
        "feature": rf_feature_names,
        "importance": rf_feature_importances,
    }
)

rf_feature_importance_df = rf_feature_importance_df.sort_values(
    "importance",
    ascending=False,
)

rf_feature_importance_df.to_csv(
    RF_TABLE_DIR / "random_forest_feature_importance.csv",
    index=False,
)

display(rf_feature_importance_df.head(30))

,feature,importance
3,num__TotalCharges,0.228045
2,num__MonthlyCharges,0.190246
1,num__tenure,0.153586
43,cat__PaymentMethod_Electronic check,0.065734
36,cat__Contract_Month-to-month,0.059881
16,cat__InternetService_Fiber optic,0.038508
18,cat__OnlineSecurity_No,0.033759
27,cat__TechSupport_No,0.022022
38,cat__Contract_Two year,0.014394
15,cat__InternetService_DSL,0.012402


In [28]:
#聚合One-Hot后的原始变量重要性
def clean_original_feature_name(feature_name: str) -> str:
    """
    Convert transformed feature name back to original feature group.

    Examples
    --------
    num__tenure -> tenure
    cat__Contract_Month-to-month -> Contract
    cat__PaymentMethod_Electronic check -> PaymentMethod
    """
    feature_name = str(feature_name)

    if feature_name.startswith("num__"):
        return feature_name.replace("num__", "", 1)

    if feature_name.startswith("cat__"):
        name = feature_name.replace("cat__", "", 1)

        matched_cols = [
            col for col in categorical_cols
            if name == col or name.startswith(f"{col}_")
        ]

        if matched_cols:
            return sorted(matched_cols, key=len, reverse=True)[0]

        return name

    return feature_name


rf_feature_importance_df["original_feature"] = rf_feature_importance_df["feature"].apply(
    clean_original_feature_name
)

rf_original_feature_importance_df = (
    rf_feature_importance_df
    .groupby("original_feature", as_index=False)["importance"]
    .sum()
    .sort_values("importance", ascending=False)
)

rf_original_feature_importance_df.to_csv(
    RF_TABLE_DIR / "random_forest_original_feature_importance.csv",
    index=False,
)

display(rf_original_feature_importance_df)

,original_feature,importance
16,TotalCharges,0.228045
4,MonthlyCharges,0.190246
18,tenure,0.153586
0,Contract,0.082093
10,PaymentMethod,0.080879
3,InternetService,0.053693
7,OnlineSecurity,0.041936
15,TechSupport,0.028451
6,OnlineBackup,0.019294
8,PaperlessBilling,0.017075


In [29]:
#比较三个模型的性能指标
metrics_files = [
    PROJECT_DIR / "reports" / "model" / "baseline" / "dummy" / "tables" / "dummy_metrics.csv",
    PROJECT_DIR / "reports" / "model" / "baseline" / "logistic_regression" / "tables" / "logistic_metrics.csv",
    RF_TABLE_DIR / "random_forest_metrics.csv",
]

metrics_dfs = []

for path in metrics_files:
    if path.exists():
        temp_df = pd.read_csv(path)
        metrics_dfs.append(temp_df)
    else:
        print("Missing metrics file:", path)

baseline_metrics_comparison_df = pd.concat(metrics_dfs, ignore_index=True)

baseline_metrics_comparison_df.to_csv(
    RF_TABLE_DIR / "baseline_metrics_comparison.csv",
    index=False,
)

display(baseline_metrics_comparison_df)

,model,strategy,positive_class,train_size,valid_size,accuracy,balanced_accuracy,precision,recall,f1,roc_auc,average_precision
0,DummyClassifier,most_frequent,Yes,475355,118839,0.774796,0.500000,0.000000,0.000000,0.000000,0.500000,0.225204
1,LogisticRegression,NaN,Yes,475355,118839,0.854416,0.784753,0.683671,0.657998,0.670589,0.908428,0.727093
2,RandomForest,NaN,Yes,475355,118839,0.840423,0.752862,0.662676,0.593543,0.626207,0.892798,0.690610
